# 03 — First features

Computes pre-fight rolling features on top of the de-biased fight table from
notebook 02, closing the post-fight statistics leakage trap: the raw
per-fight stats (strikes, takedowns, control time...) are outcomes of the
fight they're attached to, so they can't be used directly. This notebook
rolls them up per fighter, using only fights strictly before the one being
predicted, and attaches the static (time-invariant) attributes — age,
height, reach, stance — on top.

Weight is deliberately excluded: it changes across a fighter's career as
they move divisions, so a single joined value wouldn't be safe the way
reach/stance are — `weight_class` on the fight itself is the correct source
if weight is ever needed.

**Design choices.**

*All remaining raw stats rolled* — including the strike-location breakdown
(head/body/leg, distance/clinch/ground) and `reversals`. Note these are exact
decompositions of `sig_strikes` (verified: it equals head+body+leg and
distance+clinch+ground on 100% of rows), so expect collinearity.

*Attempts + accuracy instead of landed + attempted* — since
`landed = attempts × accuracy`, that's a reparameterisation rather than a
loss, but it separates "how much they throw" from "how well it lands".
Accuracy is built in both directions: the fighter's own, and how often
opponents land what they throw *at* them.

*Volume normalised by fight length, both directions* — a raw count conflates
skill with fight length. Attempt counts, knockdowns, submissions and
reversals become per-minute rates; control time becomes a **share of the
bout**, since it already shares units with fight duration. This applies to
both what a fighter does (offence) and what happens to them (defence).
`total_fight_time` is rolled too, giving each fighter's average bout length
— useful on its own as a finisher/distance-goer signal, and the denominator
these rates need.

*Reach-to-height ratio ("ape index")* — `reach_cm − height_cm`. A distinct
scouting concept from either alone: two fighters can share identical reach
but very different frames depending on height.

*Stance matchup* — `same_stance`, a single symmetric boolean rather than a
diff (stance is categorical, and "is this an open matchup" is the same fact
regardless of which fighter is labelled A or B). A directional flag assuming
southpaw or orthodox has the advantage was deliberately not built — that's a
hypothesis for the model to test, not something to hard-code.

*Prior title and 5-round fight counts* — cumulative, the same pattern as
`wins`/`losses`. Title status comes from `weight_class`; 5-round status adds
any fight (title or not) that ran past 900 seconds, since a 3-round fight
cannot physically exceed that — this is a lower bound, since a 5-round fight
that ends by an early finish is indistinguishable from a 3-round fight
ending the same way.

**Draws and no-contests are history too.** Notebook 02 keeps every
fight, so a fighter's rolled averages, experience count and layoff include
the bouts that had no winner. They are removed only when modelling rows are
chosen, by `chronological_split`. `wins`/`losses` count neither; a draw is
half a win in `win_rate` and resets the streak; a no-contest is excluded
from `win_rate` and leaves the streak alone.

**Shrinkage.** Every feature that is a total divided by an amount of evidence
— accuracy, per-minute rates, control share, win rate, average fight time — is
shrunk towards the league value, so a tiny sample can't produce an extreme
number: one knockdown in an 8-second fight is not "a knockdown every 8
seconds". The strengths (alphas) are learned from the training years only
(2014–2022) and saved to `data/processed/alphas.json`, because a model must
always be paired with the alphas its features were built with. See
`features/shrinkage.py`.

As with notebook 02, the actual transform lives in
`src/ufc_predictor/features/` (`rolling.py`, `static.py`) — this notebook
calls it and checks the result.

**Elo.** A rating built from who beat whom, the only feature that knows
how good a fighter's opponents were (`features/elo.py`). Its one setting, K, is
chosen in `05_elo.ipynb` by how much the rating helps the model, and saved to
`data/processed/elo_params.json`. This notebook adds the Elo columns whenever
that file exists. The whole chain now lives in one function,
`features/pipeline.build_featured_table`, which the comparison arms in
`modeling/variants.py` use too, so no table can silently skip a step.

**The input review.** After reviewing all 108 inputs and testing the result as a
model, the model now uses **40 chosen inputs**, listed explicitly
in `manifest.FEATURES` (`reviews/final_inputs_2026-10-08.csv` says why each one is there,
`reviews/input_review_2026-10-07.csv` why the others are not). The recent-form window is
**8 fights** instead of 3, and each rolling stat enters the model **once**, as its 8-fight
version. Career windows and the unselected inputs are still built (they are
`manifest.unselected_columns`), so nothing here is lost. Two renames: `diff_static_age` is
now `diff_static_age_gap`, and a ratio of what is thrown *at* a fighter is their **hit rate
against** (takedowns: **success rate against**), no longer "absorbed accuracy".

In [1]:
from ufc_predictor.config import PROCESSED_DIR, VALIDATION_START
from ufc_predictor.data.io import load_processed, load_fighters, save_processed
from ufc_predictor.features.elo import load_elo_params, load_elo_weights
from ufc_predictor.features.pipeline import build_featured_table
from ufc_predictor.features.rolling import fit_alphas
from ufc_predictor.features.shrinkage import save_alphas

fights = load_processed("fights")
fighters = load_fighters()

# how strongly to shrink each ratio feature - learned from the training years
# only, and saved next to the table it builds
alphas = fit_alphas(fights, until=VALIDATION_START)
save_alphas(alphas, until=VALIDATION_START)

# Elo's K is chosen in 05_elo.ipynb, by how much the rating helps the model.
# Until that has run once, the table is built without the Elo columns.
elo_k = load_elo_params() if (PROCESSED_DIR / "elo_params.json").exists() else None
elo_weights = load_elo_weights() if elo_k is not None else None

# the whole chain - prior features, static attributes, weight class, Elo - in
# one shared function, the same one the comparison arms in modeling/variants.py use
featured = build_featured_table(fights, fighters, alphas, elo_k=elo_k, elo_weights=elo_weights)
featured.shape

(8937, 374)

## Shrinkage strengths

`shrunk = (total + α × league) / (evidence + α)`. α is the amount of imaginary
league-average evidence every fighter starts with, counted in the feature's own
unit: attempts for accuracy, minutes for rates and control share, fights for win
rate and average fight time. Each feature's α was chosen by testing which value
best predicts what fighters did in their next fight (for win rate: who won),
using the training years only (2014–2022).

In [2]:
from ufc_predictor.features.rolling import RATIO_FEATURES
from ufc_predictor.features.shrinkage import describe_alphas

table = describe_alphas(RATIO_FEATURES, alphas)
print(f"{len(table)} shrunk features; the career versions:")
table[table["feature"].str.contains("career")].sort_values(["unit", "alpha"]).reset_index(drop=True)

91 shrunk features; the career versions:


,feature,alpha,unit
0,takedowns_success_rate_against_career_avg,31.620,attempts
1,takedowns_accuracy_career_avg,100.000,attempts
2,leg_strikes_accuracy_career_avg,125.900,attempts
3,leg_strikes_hit_rate_against_career_avg,125.900,attempts
4,body_strikes_accuracy_career_avg,158.500,attempts
5,clinchs_accuracy_career_avg,158.500,attempts
6,body_strikes_hit_rate_against_career_avg,199.500,attempts
7,grounds_hit_rate_against_career_avg,199.500,attempts
8,clinchs_hit_rate_against_career_avg,251.200,attempts
9,dist_strikes_accuracy_career_avg,398.100,attempts


## Sanity checks

In [3]:
# no fights lost or duplicated by the merge
assert len(featured) == len(fights)

# a genuine debut fighter has no rolled-up history to draw on
assert featured["fighter_a_prior_is_debut"].sum() > 0
debut_rows = featured[featured["fighter_a_prior_is_debut"]]
assert debut_rows["fighter_a_prior_sig_strike_atts_per_min_career_avg"].isna().all()
assert debut_rows["fighter_a_prior_sig_strikes_accuracy_career_avg"].isna().all()
assert debut_rows["fighter_a_prior_total_fight_time_career_avg"].isna().all()

# shrinkage: a veteran's ratio features are never NaN any more - a fighter who
# never attempted something gets the league value. (A few veterans in 1994-97
# are NaN: the early UFC didn't record every stat, so there was no league value
# to shrink towards. None of them are in the modelling era.)
from ufc_predictor.config import ERA_START
from ufc_predictor.features.rolling import RATIO_FEATURES
era_veterans = featured[(featured["event_date"] >= ERA_START) & ~featured["fighter_a_prior_is_debut"]]
ratio_cols = [f"fighter_a_prior_{spec.name}" for spec in RATIO_FEATURES]
assert era_veterans[ratio_cols].notna().all().all()

# undecided fights (draws and no-contests) are present, carry no label - and
# only they lack one - and still have features
undecided = featured["outcome"].isin(["draw", "no_contest"])
assert undecided.sum() > 0
assert featured.loc[undecided, "fighter_a_won"].isna().all()
assert featured.loc[~undecided, "fighter_a_won"].notna().all()
assert featured.loc[undecided, "diff_prior_fight_count"].notna().any()
print(f"undecided fights kept as history: {undecided.sum()}")

any_debut_rate = (
    featured["fighter_a_prior_is_debut"] | featured["fighter_b_prior_is_debut"]
).mean()
print(f"fights involving at least one debutant: {any_debut_rate:.1%}")

undecided fights kept as history: 157
fights involving at least one debutant: 25.2%


In [4]:
# leakage spot-check: Jim Miller's 3rd UFC fight (2009-03-07) should show a
# prior record of 2-0, not his eventual career record — this is the exact
# check that verified the module against a hand-computed example.
#
# he lands in the fighter_a slot for some fights and fighter_b for others
# (that's the point of the de-biasing in notebook 02), so both sides need
# checking, not just fighter_a. Found by name: ids depend on the data source.
import pandas as pd

named = pd.concat([featured[["fighter_a_id", "fighter_a_name"]].set_axis(["id", "name"], axis=1),
                   featured[["fighter_b_id", "fighter_b_name"]].set_axis(["id", "name"], axis=1)])
jim_miller_ids = named.loc[named["name"] == "Jim Miller", "id"].unique()
assert len(jim_miller_ids) == 1, "expected exactly one Jim Miller"
jim_miller_id = jim_miller_ids[0]

mask = (featured["fighter_a_id"] == jim_miller_id) | (featured["fighter_b_id"] == jim_miller_id)
his_fights = featured[mask].sort_values("event_date")
his_3rd_fight = his_fights.iloc[2]
side = "fighter_a" if his_3rd_fight["fighter_a_id"] == jim_miller_id else "fighter_b"

prior_wins = his_3rd_fight[f"{side}_prior_wins"]
prior_losses = his_3rd_fight[f"{side}_prior_losses"]
print("date:", his_3rd_fight["event_date"].date())
print("prior_wins:", prior_wins)
print("prior_losses:", prior_losses)
assert his_3rd_fight["event_date"] == pd.Timestamp("2009-03-07")
assert prior_wins == 2
assert prior_losses == 0

date: 2009-03-07
prior_wins: 2.0
prior_losses: 0.0


In [5]:
# static feature sanity checks: row count preserved, diffs are simple
# subtraction, missingness matches what we already knew about the raw
# fighter table (reach ~45%, stance ~19%) rather than something worse
assert len(featured) == len(fights)

diff_pairs = {
    "diff_static_age_gap": ("fighter_a_age_during", "fighter_b_age_during"),
    "diff_static_height_cm": ("fighter_a_height_cm", "fighter_b_height_cm"),
    "diff_static_reach_cm": ("fighter_a_reach_cm", "fighter_b_reach_cm"),
}
for diff_col, (a_col, b_col) in diff_pairs.items():
    assert featured[diff_col].equals(featured[a_col] - featured[b_col])

print("reach missing:", f"{featured['fighter_a_reach_cm'].isna().mean():.1%}")
print("stance missing:", f"{featured['fighter_a_stance'].isna().mean():.1%}")
print("age missing:", f"{featured['fighter_a_age_during'].isna().mean():.1%}")

reach missing: 7.4%
stance missing: 0.5%
age missing: 0.8%


In [6]:
# accuracy sanity checks: ratios must sit in [0, 1], and the defensive
# direction must carry information the raw absorbed volume doesn't — that
# near-zero correlation is the whole reason it was added.
#
# note the bound applies to the per-fighter columns only: diff_* columns are
# differences of two accuracies, so they legitimately span [-1, 1].
per_fighter_acc = [
    c for c in featured.columns
    if ("_accuracy_" in c or "_rate_against_" in c) and c.startswith(("fighter_a_", "fighter_b_"))
]
print(f"{len(per_fighter_acc)} per-fighter accuracy columns")

for col in per_fighter_acc:
    vals = featured[col].dropna()
    assert vals.between(0, 1).all(), f"{col} outside [0,1]: {vals.min()}–{vals.max()}"

# control time absorbed is a share of the bout, so it cannot exceed 1.0.
# it did before the denominator was fixed to cover only the bouts that
# actually recorded control time — see features/shrinkage.py.
for col in [c for c in featured.columns
            if "_frac_" in c and c.startswith(("fighter_a_", "fighter_b_"))]:
    vals = featured[col].dropna()
    assert vals.between(0, 1).all(), f"{col} outside [0,1]: {vals.min()}–{vals.max()}"

both = featured[[
    "fighter_a_prior_sig_strike_atts_absorbed_per_min_career_avg",
    "fighter_a_prior_sig_strikes_hit_rate_against_career_avg",
]].dropna()
corr = both.corr().iloc[0, 1]
print(f"corr(absorbed volume/min, opponent accuracy against fighter): {corr:.3f}")

print("\nopponent accuracy against fighter (how hittable they are):")
print(featured["fighter_a_prior_sig_strikes_hit_rate_against_career_avg"].describe()[
    ["mean", "std", "min", "max"]
].round(3).to_string())

print("\nshare of bout spent controlled by the opponent:")
print(featured["fighter_a_prior_ctrl_time_absorbed_frac_career_avg"].describe()[
    ["mean", "std", "min", "max"]
].round(3).to_string())

72 per-fighter accuracy columns
corr(absorbed volume/min, opponent accuracy against fighter): -0.062

opponent accuracy against fighter (how hittable they are):
mean    0.439
std     0.035
min     0.316
max     0.606

share of bout spent controlled by the opponent:
mean    0.209
std     0.061
min     0.057
max     0.457


In [7]:
# item 1: ape index. real spread, and now a feature rather than left implicit
# in reach_cm and height_cm separately.
ape = featured["fighter_a_ape_index_cm"].dropna()
print(f"ape index (reach - height): mean {ape.mean():.1f}cm, std {ape.std():.1f}, "
      f"range {ape.min():.1f} to {ape.max():.1f}")

# item 3: stance matchup. a symmetric fact, not diffed.
print(f"\nsame_stance dtype: {featured['same_stance'].dtype}")
print(featured["same_stance"].value_counts(dropna=False))

# item 4: title / five-round counts are cumulative and monotonic per fighter,
# same shape as wins/losses. sanity-check against a known multi-title veteran.
top = featured.loc[featured["fighter_a_prior_title_fights"].idxmax()]
print(f"\nmost prior UFC title fights (fighter_a slot): {top['fighter_a_prior_title_fights']:.0f} "
      f"— {top['fighter_a_name']} on {top['event_date'].date()}")

assert featured["fighter_a_prior_five_round_fights"].ge(
    featured["fighter_a_prior_title_fights"]
).all(), "every title fight is 5-round, so five_round_fights can't be less than title_fights"

ape index (reach - height): mean 4.6cm, std 4.9, range -12.7 to 22.9

same_stance dtype: boolean
same_stance
True     5377
False    3472
<NA>       88
Name: count, dtype: Int64

most prior UFC title fights (fighter_a slot): 16 — Randy Couture on 2009-08-29


## A look at the engineered features

In [8]:
diff_cols = [c for c in featured.columns if c.startswith("diff_prior_") or c.startswith("diff_static_")]
print(f"{len(diff_cols)} diff features")
featured[diff_cols].describe().T[["mean", "std", "min", "max"]]

103 diff features


,mean,std,min,max
diff_prior_fight_count,0.081347,6.728125,-41.000000,37.000000
diff_prior_wins,0.044758,4.123188,-24.000000,21.000000
diff_prior_losses,0.042184,2.926263,-16.000000,16.000000
diff_prior_win_rate,-0.001222,0.115410,-0.417614,0.478746
diff_prior_streak,-0.028309,2.356248,-19.000000,13.000000
...,...,...,...,...
diff_static_age_gap,0.062762,5.303569,-17.000000,23.000000
diff_static_height_cm,-0.011685,6.450175,-33.020000,27.940000
diff_static_reach_cm,0.007089,8.286190,-30.480000,33.020000
diff_static_ape_index_cm,-0.029329,6.588681,-25.400000,25.400000


## Save

In [9]:
save_processed(featured, "fights_featured")
featured.head()

,weight_class,method,fights_url,event_url,event_date,fighter_a_age_during,fighter_b_age_during,fighter_a_body_strike_atts,fighter_b_body_strike_atts,fighter_a_body_strikes,...,diff_static_height_cm,diff_static_reach_cm,diff_static_ape_index_cm,same_stance,division,weight_class_lbs,is_womens_bout,fighter_a_prior_elo,fighter_b_prior_elo,diff_prior_elo
0,Catch Weight Bout,finish,http://www.ufcstats.com/fight-details/4acab678...,http://www.ufcstats.com/event-details/a6a9ab5a...,1994-03-11,18.0,NaN,0.0,0.0,0.0,...,5.08,NaN,NaN,<NA>,no_division,NaN,False,1500.00,1500.000000,0.000000
1,UFC Catch Weight Title Bout,finish,http://www.ufcstats.com/fight-details/00835554...,http://www.ufcstats.com/event-details/a6a9ab5a...,1994-03-11,27.0,30.0,0.0,2.0,0.0,...,-2.54,NaN,NaN,False,no_division,NaN,False,1500.00,1500.000000,0.000000
2,Catch Weight Bout,finish,http://www.ufcstats.com/fight-details/aa161c71...,http://www.ufcstats.com/event-details/a6a9ab5a...,1994-03-11,24.0,27.0,0.0,0.0,0.0,...,5.08,NaN,NaN,True,no_division,NaN,False,1579.25,1640.730209,-61.480209
3,Catch Weight Bout,finish,http://www.ufcstats.com/fight-details/8fbdde20...,http://www.ufcstats.com/event-details/a6a9ab5a...,1994-03-11,NaN,30.0,0.0,2.0,0.0,...,-5.08,NaN,NaN,True,no_division,NaN,False,1500.00,1610.898854,-110.898854
4,Catch Weight Bout,finish,http://www.ufcstats.com/fight-details/5655639b...,http://www.ufcstats.com/event-details/a6a9ab5a...,1994-03-11,24.0,27.0,0.0,0.0,0.0,...,-5.08,NaN,NaN,True,no_division,NaN,False,1500.00,1579.250000,-79.250000


## Visualise the dataset

The full table is far too wide to read directly. A curated view of the most
recent fights: who fought, the label, and the headline engineered features —
striking volume, accuracy in both directions, how much of the bout they
spend controlled, and typical bout length.

In [10]:
readable_cols = [
    "event_date", "weight_class",
    "fighter_a_name", "fighter_b_name", "fighter_a_won",
    "diff_prior_win_rate", "diff_prior_streak",
    "diff_prior_sig_strike_atts_per_min_career_avg",       # striking volume, per minute
    "diff_prior_sig_strikes_accuracy_career_avg",          # striking accuracy (offence)
    "diff_prior_sig_strikes_hit_rate_against_career_avg",  # how hittable they are
    "diff_prior_ctrl_time_frac_career_avg",                # share of bout THEY controlled
    "diff_prior_title_fights",                             # title-fight experience
    "diff_static_age_gap", "diff_static_reach_cm", "diff_static_ape_index_cm",
    "same_stance",
]

view = featured[readable_cols].sort_values("event_date", ascending=False)
view.head(20)

,event_date,weight_class,fighter_a_name,fighter_b_name,fighter_a_won,diff_prior_win_rate,diff_prior_streak,diff_prior_sig_strike_atts_per_min_career_avg,diff_prior_sig_strikes_accuracy_career_avg,diff_prior_sig_strikes_hit_rate_against_career_avg,diff_prior_ctrl_time_frac_career_avg,diff_prior_title_fights,diff_static_age_gap,diff_static_reach_cm,diff_static_ape_index_cm,same_stance
8936,2026-10-10,Middleweight Bout,Brendan Allen,Christian Leroy Duncan,True,0.024751,-2.0,-0.513803,-0.028192,0.032084,0.081219,0.0,-1.0,-10.16,-1.016000e+01,False
8930,2026-10-10,Light Heavyweight Bout,Felipe Franco,Brendson Ribeiro,False,0.166931,5.0,-0.025014,0.044912,0.000908,0.089663,0.0,-5.0,-10.16,-5.080000e+00,True
8925,2026-10-10,Lightweight Bout,Jai Herbert,Matheus Camilo,True,-0.107386,-1.0,0.766892,-0.027487,0.024677,-0.034200,0.0,13.0,20.32,1.270000e+01,True
8926,2026-10-10,Women's Strawweight Bout,Loopy Godinez,Ketlen Souza,True,0.033493,-3.0,1.242339,-0.031876,-0.064925,0.069918,0.0,1.0,-5.08,-2.540000e+00,True
8927,2026-10-10,Featherweight Bout,Kai Kamaka III,Andre Fili,False,-0.050095,1.0,-1.104125,0.149196,-0.033979,0.047516,0.0,-5.0,-12.70,-2.540000e+00,True
8929,2026-10-10,Light Heavyweight Bout,Gerald Meerschaert,Julius Walker,False,0.091943,-3.0,-0.807599,-0.039015,-0.027788,-0.141215,0.0,11.0,-2.54,5.080000e+00,False
8928,2026-10-10,Bantamweight Bout,Otari Tanzilovi,Malcolm Wellmaker,True,-0.100381,1.0,-0.529057,0.007794,0.007279,0.085092,0.0,-4.0,-2.54,2.842171e-14,False
8931,2026-10-10,Welterweight Bout,Niko Price,Leon Shahbazyan,True,0.035110,-3.0,2.922709,-0.032586,0.026345,-0.030436,0.0,7.0,5.08,1.524000e+01,True
8932,2026-10-10,Heavyweight Bout,Allen Frye Jr.,RJ Harris,True,-0.200763,-2.0,-0.987454,-0.018369,0.015872,0.096606,0.0,0.0,5.08,-2.842171e-14,True
8933,2026-10-10,Women's Bantamweight Bout,Daria Zhelezniakova,Alice Pereira,False,0.000000,-2.0,1.330669,-0.007254,-0.009577,-0.030292,0.0,10.0,-7.62,-1.016000e+01,True


In [11]:
# load the saved wide table back in a fresh session, for inspection
from ufc_predictor.data.io import load_processed

df = load_processed("fights_featured")
df.shape

(8937, 374)

## The X/y manifest

The table's columns are not interchangeable. 46 of them are the raw per-fight stats
of *the bout being predicted*, sitting one word away from their safe
counterparts — `fighter_a_sig_strikes` is the answer sheet,
`fighter_a_prior_sig_strikes_accuracy_career_avg` is a feature. So `X` gets
built from an explicit allow-list, never by dropping columns.

`manifest.py` groups the safe features by what they measure, so the choice of
what to train on can be made by measurement rather than up front. Two guards
back it: no raw post-fight column can be selected, and every column in the
table must be classified — add one without deciding what it is and the test
suite fails rather than letting it drift silently into or out of `X`.

In [12]:
from ufc_predictor.features import manifest

# completeness guard: nothing in the table is unaccounted for
unclassified = manifest.unclassified_columns(featured)
assert unclassified == [], f"unclassified columns: {unclassified}"

# make_xy refuses rows with no winner, so go through the split, which drops
# draws/no-contests and keeps only the 2014+ era
from ufc_predictor.modeling.split import chronological_split

train, validation, test = chronological_split(featured)
print(f"train {len(train):,}   validation {len(validation):,}   test {len(test):,}")

X, y = manifest.make_xy(train)
print(f"X: {X.shape[0]} rows x {X.shape[1]} features     y: {y.shape[0]} rows")
print(f"leaky columns held out of X: {len(manifest.leaky_columns(featured))}")
print(f"label in X? {manifest.LABEL_COLUMN in X.columns}")

train 4,309   validation 1,017   test 950
X: 4309 rows x 40 features     y: 4309 rows
leaky columns held out of X: 47
label in X? False


In [13]:
import pandas as pd
# What each feature group costs in missing rows.
#
# Measured on the modelling rows only (2014+, decided).
# Nothing here is imputed - the remaining NaNs are real.
report = manifest.nan_report(pd.concat([train, validation, test]))
report["rows_with_nan"] = (report["rows_with_nan"] * 100).round(1)
print(report.to_string(index=False))

print(
    "\nAfter shrinkage a missing value means one of two things: a debutant (no UFC"
    "\nhistory - genuinely unknown, left for the model to handle) or a gap in the raw"
    "\nfighter table (reach, stance, height). The 0/0 accuracy NaNs are gone: a fighter"
    "\nwho never attempted something now gets the league value."
)

     group  columns  rows_with_nan
  strength        1            0.0
cold_start        2            0.0
      pace        0            0.0
  physical        5            4.9
    record        3           21.1
  striking       13           21.1
 grappling        4           21.1
    damage        1           21.1
   defence       11           21.1
       ALL       40           23.3

After shrinkage a missing value means one of two things: a debutant (no UFC
history - genuinely unknown, left for the model to handle) or a gap in the raw
fighter table (reach, stance, height). The 0/0 accuracy NaNs are gone: a fighter
who never attempted something now gets the league value.
